# Product Classification & Clustering
## Unsupervised Learning for Product Segmentation
This project applies unsupervised machine learning techniques to analyze and segment products based on product, merchant, category, and cluster-related attributes.
The project explores multiple feature representations and clustering approaches, including PCA, K-Means, DBSCAN, and an Autoencoder-based latent representation.
The goal is to identify meaningful product groups and evaluate how different representations affect clustering performance.
## Project Objectives
- Explore the structure and characteristics of the product dataset.
- Identify and prepare suitable features for unsupervised learning.
- Apply appropriate preprocessing and feature scaling techniques.
- Explore the data using PCA.
- Build a K-Means clustering baseline.
- Use an Autoencoder to learn a lower-dimensional latent representation.
- Apply clustering to the learned latent features.
- Compare clustering performance across different approaches.
- Analyze and interpret the resulting product segments.

## 1. Project Overview
This project analyzes product data using unsupervised machine learning techniques.
The workflow combines categorical feature engineering, K-Means clustering, Autoencoder-based representation learning, and DBSCAN to identify structure within product data.

In [ ]:
%pip install tensorflow

: 

In [ ]:
%pip install numpy pandas matplotlib scikit-learn tensorflow

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense

print("All libraries imported successfully.")

ModuleNotFoundError: No module named 'tensorflow'

: 

## 2. Data Loading & Initial Exploration

The dataset is loaded and inspected to understand its structure, dimensions, and available product attributes.

In [2]:
import urllib.request
import zipfile
import os
# Download dataset
url = "https://archive.ics.uci.edu/static/public/837/product+classification+and+clustering.zip"
zip_path = "product_classification_clustering.zip"
urllib.request.urlretrieve(url, zip_path)
# Extract dataset
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("data")
print("Dataset downloaded and extracted successfully.")

Dataset downloaded and extracted successfully.


In [ ]:
for root, dirs, files in os.walk("data"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
# Load dataset
csv_path = "data/pricerunner_aggregate.csv"
df = pd.read_csv(csv_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
# Display the first 5 rows
df.head()

## 3. Exploratory Data Analysis
Before preprocessing the data, we examine the dataset structure, data types, missing values, and duplicated records.

In [ ]:
# Exploratory Data Analysis

print("===== DATASET INFO =====")
df.info()

print("\n===== MISSING VALUES =====")
print(df.isnull().sum())

print("\n===== DUPLICATE ROWS =====")
print("Duplicate rows:", df.duplicated().sum())

print("\n===== UNIQUE VALUES =====")
print(df.nunique())

In [ ]:
# Normalize column names
df.columns = [c.strip() for c in df.columns]

# Find category and merchant columns case-insensitively
cat_col = next((c for c in df.columns if "category" in c.lower()), None)
merchant_col = next((c for c in df.columns if "merchant" in c.lower()), None)

print("Category column:", cat_col)
print("Merchant column:", merchant_col)

if cat_col is None or merchant_col is None:
    raise ValueError(
        f"Could not find category/merchant columns. "
        f"Available columns: {list(df.columns)}"
    )

# Visualize category and merchant distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df[cat_col].value_counts(dropna=False).head(15).sort_values().plot(
    kind="barh", ax=axes[0]
)
axes[0].set_title("Top 15 Product Categories")
axes[0].set_xlabel("Number of Products")
axes[0].set_ylabel("Category")

df[merchant_col].value_counts(dropna=False).head(15).sort_values().plot(
    kind="barh", ax=axes[1]
)
axes[1].set_title("Top 15 Merchants by Number of Products")
axes[1].set_xlabel("Number of Products")
axes[1].set_ylabel("Merchant")

plt.tight_layout()
plt.show()

## 4. Feature Engineering & Preprocessing

In [ ]:
# Select features for clustering

print("Available columns:")
print(df.columns.tolist())

In [ ]:
# Feature Engineering
# Exclude identifiers, product text, and existing cluster labels
# to avoid data leakage.

feature_df = df[["Merchant ID", "Category ID"]].copy()

# Convert categorical IDs to numeric values
feature_df["Merchant ID"] = pd.to_numeric(
    feature_df["Merchant ID"], errors="coerce"
)
feature_df["Category ID"] = pd.to_numeric(
    feature_df["Category ID"], errors="coerce"
)

# Remove rows with missing feature values
feature_df = feature_df.dropna().reset_index(drop=True)

print("Features used for clustering:")
print(feature_df.columns.tolist())

print("\nFeature shape:", feature_df.shape)
print("\nMissing values:")
print(feature_df.isnull().sum())

In [ ]:
feature_df.head()

In [ ]:
# Feature Engineering

feature_df = df[["Merchant ID", "Category ID"]].copy()

# Treat IDs as categorical features
feature_df = feature_df.astype(str)

# One-hot encode categorical features
X = pd.get_dummies(
    feature_df,
    columns=["Merchant ID", "Category ID"],
    dtype=float
)

print("Encoded feature shape:", X.shape)
print("Missing values:", X.isnull().sum().sum())

X.head()

## 5. Dimensionality Reduction

The one-hot encoded feature space contains 316 dimensions. PCA is applied to reduce dimensionality, preserve the main sources of variation, and provide a compact representation for visualization and subsequent clustering analysis.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)

# PCA for dimensionality reduction
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("PCA shape:", X_pca.shape)
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2%}")
plt.figure(figsize=(10, 7))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    s=8,
    alpha=0.5
)

plt.title("PCA Projection of Product Features")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

In [ ]:
# PCA explained variance analysis

pca_full = PCA()
pca_full.fit(X_scaled)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components needed to retain 80%, 90%, and 95% variance
for threshold in [0.80, 0.90, 0.95]:
    n_components = np.argmax(cumulative_variance >= threshold) + 1
    print(f"{threshold:.0%} variance: {n_components} components")

# Plot cumulative explained variance
plt.figure(figsize=(10, 6))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance
)

plt.axhline(0.80, linestyle="--", label="80%")
plt.axhline(0.90, linestyle="--", label="90%")
plt.axhline(0.95, linestyle="--", label="95%")

plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

## 6. K-Means Clustering Baseline

K-Means is first applied to the standardized feature space as a baseline clustering method. The optimal number of clusters is evaluated using the Silhouette Score.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Evaluate different numbers of clusters
k_values = range(2, 11)
silhouette_scores = []

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    
    print(f"K={k}: Silhouette Score = {score:.4f}")

# Find the best K
best_k = k_values[np.argmax(silhouette_scores)]

print(f"\nBest K: {best_k}")
print(f"Best Silhouette Score: {max(silhouette_scores):.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means: Silhouette Score by Number of Clusters")
plt.xticks(list(k_values))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Autoencoder-Based Representation Learning

The baseline clustering results indicate that the original one-hot feature space does not provide well-separated clusters.

An Autoencoder is therefore used to learn a compact latent representation of the product features. The learned representation will then be used as input for clustering.

In [ ]:
# Autoencoder

from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense

input_dim = X_scaled.shape[1]
latent_dim = 16

# Encoder
inputs = Input(shape=(input_dim,))
encoded = Dense(128, activation="relu")(inputs)
encoded = Dense(64, activation="relu")(encoded)
latent = Dense(latent_dim, activation="relu", name="latent_space")(encoded)

# Decoder
decoded = Dense(64, activation="relu")(latent)
decoded = Dense(128, activation="relu")(decoded)
outputs = Dense(input_dim, activation="linear")(decoded)

# Autoencoder model
autoencoder = Model(inputs, outputs)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

In [ ]:
# Train Autoencoder

history = autoencoder.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)

In [ ]:
# Autoencoder training loss

plt.figure(figsize=(10, 6))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Latent Representation

The trained encoder is used to transform the original feature space into a compact 16-dimensional latent representation.

In [ ]:
encoder = Model(
    inputs=autoencoder.input,
    outputs=autoencoder.get_layer("latent_space").output
)

X_latent = encoder.predict(X_scaled, verbose=0)

print("Latent representation shape:", X_latent.shape)

## 8. K-Means Clustering on Latent Space

In [ ]:
# K-Means on Autoencoder latent representation

latent_k_values = range(2, 11)
latent_silhouette_scores = []

for k in latent_k_values:
    kmeans_latent = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    latent_labels = kmeans_latent.fit_predict(X_latent)

    score = silhouette_score(X_latent, latent_labels)
    latent_silhouette_scores.append(score)

    print(f"K={k}: Silhouette Score = {score:.4f}")

best_latent_k = latent_k_values[
    np.argmax(latent_silhouette_scores)
]

best_latent_score = max(latent_silhouette_scores)

print(f"\nBest K: {best_latent_k}")
print(f"Best Silhouette Score: {best_latent_score:.4f}")

In [ ]:
# Compare clustering performance

plt.figure(figsize=(10, 6))

plt.plot(
    list(latent_k_values),
    latent_silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means on Autoencoder Latent Space")
plt.xticks(list(latent_k_values))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Final K-Means clustering on Autoencoder latent space

final_kmeans = KMeans(
    n_clusters=best_latent_k,
    random_state=42,
    n_init=10
)

final_labels = final_kmeans.fit_predict(X_latent)

print("Final clustering completed.")
print("Number of clusters:", best_latent_k)
print("Silhouette Score:", f"{best_latent_score:.4f}")

print("\nCluster distribution:")
print(pd.Series(final_labels).value_counts().sort_index())

In [ ]:
# Visualize clusters using the first two latent dimensions

plt.figure(figsize=(10, 7))

plt.scatter(
    X_latent[:, 0],
    X_latent[:, 1],
    c=final_labels,
    s=10,
    alpha=0.6
)

plt.title("Product Clusters in Autoencoder Latent Space")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.tight_layout()
plt.show()

## 9. Cluster Analysis & Insights

The final K-Means clusters are analyzed using product categories and merchant information to better understand the characteristics and distribution of each cluster.

In [ ]:
# Analyze final clusters

analysis_df = df.copy()
analysis_df["Cluster"] = final_labels

print("=== Cluster Distribution ===")
print(
    analysis_df["Cluster"]
    .value_counts()
    .sort_index()
    .to_frame("Product Count")
)

print("\n=== Cluster Percentage ===")
print(
    (analysis_df["Cluster"].value_counts(normalize=True) * 100)
    .sort_index()
    .round(2)
    .to_frame("Percentage")
)

print("\n=== Top Categories by Cluster ===")
for cluster in sorted(analysis_df["Cluster"].unique()):
    print(f"\nCluster {cluster}:")
    print(
        analysis_df.loc[
            analysis_df["Cluster"] == cluster,
            "Category Label"
        ]
        .value_counts()
        .head(10)
    )

In [ ]:
print("=== Top Merchants by Cluster ===")

for cluster in sorted(analysis_df["Cluster"].unique()):
    print(f"\nCluster {cluster}:")
    print(
        analysis_df.loc[
            analysis_df["Cluster"] == cluster,
            "Merchant ID"
        ]
        .value_counts()
        .head(10)
    )

In [ ]:
# Cluster profile

cluster_profile = (
    analysis_df
    .groupby("Cluster")
    .agg(
        Product_Count=("Product ID", "count"),
        Unique_Categories=("Category ID", "nunique"),
        Unique_Merchants=("Merchant ID", "nunique")
    )
)

cluster_profile["Percentage"] = (
    cluster_profile["Product_Count"]
    / len(analysis_df) * 100
).round(2)

cluster_profile

In [ ]:
# Top 10 categories for each cluster

for cluster in sorted(analysis_df["Cluster"].unique()):
    print(f"\n===== CLUSTER {cluster} =====")
    
    top_categories = (
        analysis_df[analysis_df["Cluster"] == cluster]["Category Label"]
        .value_counts()
        .head(10)
    )
    
    print(top_categories)

## 10. DBSCAN Clustering

In [ ]:
# DBSCAN clustering on Autoencoder latent space

from sklearn.cluster import DBSCAN

dbscan = DBSCAN(
    eps=0.5,
    min_samples=10
)

dbscan_labels = dbscan.fit_predict(X_latent)

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = np.sum(dbscan_labels == -1)

print("DBSCAN Results")
print("Number of clusters:", n_clusters_dbscan)
print("Noise points:", n_noise)

In [ ]:
# Evaluate DBSCAN clustering

mask = dbscan_labels != -1

if n_clusters_dbscan >= 2:
    dbscan_silhouette = silhouette_score(
        X_latent[mask],
        dbscan_labels[mask]
    )
    print(f"DBSCAN Silhouette Score: {dbscan_silhouette:.4f}")
else:
    dbscan_silhouette = None
    print("Silhouette Score cannot be calculated.")

## 11. Model Comparison

In [ ]:
# Final model comparison

comparison = pd.DataFrame({
    "Method": [
        "K-Means",
        "K-Means + Autoencoder",
        "DBSCAN + Autoencoder"
    ],
    "Feature Space": [
        "Original",
        "16D Latent Space",
        "16D Latent Space"
    ],
    "Clusters": [
        best_k,
        best_latent_k,
        n_clusters_dbscan
    ],
    "Silhouette Score": [
        max(silhouette_scores),
        best_latent_score,
        dbscan_silhouette
    ]
})

comparison

## 12. Conclusion

The clustering experiments show that the original feature space does not provide well-separated product groups using K-Means, achieving a best Silhouette Score of -0.0323.

The Autoencoder significantly improved the feature representation by reducing the original 316-dimensional feature space to 16 latent dimensions. K-Means on this latent representation achieved a Silhouette Score of 0.7475 with 2 clusters.

DBSCAN achieved a Silhouette Score of 0.8458 on the same latent representation and identified 265 density-based clusters, with 927 observations classified as noise.

Overall, the results demonstrate that Autoencoder-based representation learning can substantially improve the separability of product data for unsupervised clustering.